# Глава 9 — Multi-Modal Understanding

Мультимодальная модель получает не только текст. Для изображения **vision encoder** строит числовые представления, **connector** согласует их с представлениями LLM, а модель использует их вместе с текстовым вопросом. Сборку image tokens выполняет сервер модели; наш код передаёт картинку через API.

Кодовый пример главы добавляет `MultimodalMemory` и параметр `TinyAgent.run(..., image_data=...)`. Обычные текстовые запросы продолжают работать.

В книге используется обложка с животным. Здесь — собственная картинка с геометрическими фигурами, включённая в репозиторий. Пример не зависит от скачивания внешнего изображения и не проверяет знание обложки из обучающих данных.

Требуется запущенный Ollama с vision-моделью. По умолчанию используется уже установленная `gemma4:e4b`; notebook не скачивает модели.

In [ ]:
import base64
import math
from pathlib import Path
from pprint import pprint
import re
import socket

from IPython.display import Image, display
from agent import TinyAgent
from llm import LLM
from memory import MultimodalMemory
from planning import NativeReAct
from toolbox import multiply
from tools import NativeTools
from illustrated_agents.utils import TrajectoryViewer

socket.setdefaulttimeout(180)
llm = LLM(model="gemma4:e4b", think=True, temperature=0)
image_path = Path("examples/vision/shapes.png")
image_data = base64.b64encode(image_path.read_bytes()).decode("ascii")
display(Image(filename=str(image_path)))

## 1. Без картинки

Спросим об изображении без передачи изображения. Модель может попросить картинку, а может ошибочно придумать содержимое: конкретную формулировку отказа мы не считаем обязательной. Наличие правильного текста само по себе не доказывает, что модель видела картинку.

In [ ]:
agent = TinyAgent(
    llm, memory=MultimodalMemory(), tools=NativeTools(), planner=NativeReAct(max_steps=4),
)
question = "Describe the shapes from left to right, giving each color and shape. Reply in English in one sentence."
without_image = agent.run(question)
print("Без изображения:", without_image)

## 2. Тот же вопрос с изображением

Передаём байты PNG как base64. `MultimodalMemory` собирает два блока `content`: `image_url` с data URL и `text` с вопросом. Это не путь к файлу: файл сначала нужно прочитать и закодировать.

In [ ]:
answer = agent.run(question, image_data=image_data)
print("С изображением:", answer)
normalized = re.sub(r"[*_`]+", "", answer.lower())
for phrase in ("red square", "blue circle", "green triangle"):
    assert phrase in normalized, answer
positions = [normalized.index(shape) for shape in ("square", "circle", "triangle")]
assert positions == sorted(positions)
print("Проверены цвета, фигуры и их порядок.")

## 3. Картинка остаётся в истории

Следующий вопрос не передаёт `image_data` повторно, но прошлое сообщение с картинкой остаётся в памяти и снова отправляется модели. Это история текущего диалога, а не обучение модели.

In [ ]:
followup = agent.run("What color was the circle in the previous image? Answer with one color word in English.")
print(followup)
assert "blue" in followup.lower()
print(agent.trajectory.runs)
TrajectoryViewer(agent.trajectory)

## Формат сообщения

Показываем копию истории с сокращённым base64, чтобы она была читаемой. Реальная память содержит полный data URL. В траектории сохраняются текст вопроса и шаги; байты изображения туда не копируются, поэтому TrajectoryViewer не показывает исходную картинку.

In [ ]:
preview = agent.memory.get_messages()
for message in preview:
    if isinstance(message["content"], list):
        for block in message["content"]:
            if block["type"] == "image_url":
                block["image_url"]["url"] = "data:image/png;base64,<image bytes omitted from preview>"
pprint(preview)
assert agent.memory.get_messages()[3]["content"][0]["image_url"]["url"].endswith(image_data)

## 4. Изображение + инструмент + цикл агента

Используем отдельного агента: он должен посчитать фигуры на картинке и умножить их количество на 4 через инструмент. Проверяем фактически выполненный вызов и observation, а не только финальное число.

In [ ]:
tools = NativeTools()
tools.add_tool("multiply", multiply)
counting_agent = TinyAgent(llm, MultimodalMemory(), tools, NativeReAct(max_steps=4))
counted_answer = counting_agent.run(
    "Count the geometric shapes visible in this image. Use the multiply tool to multiply "
    "that count by 4, then state the numerical result in English.",
    image_data=f"data:image/png;base64,{image_data}",
)
print(counted_answer)
steps = counting_agent.trajectory.runs[0]["steps"]
actions = [step for step in steps if step.action]
assert len(actions) == 1 and actions[0].action["tool"] == "multiply"
assert sorted(float(actions[0].action["kwargs"][key]) for key in ("a", "b")) == [3.0, 4.0]
assert math.isclose(float(actions[0].observation), 12.0, rel_tol=0, abs_tol=1e-9)
assert steps[-1].observation is None and steps[-1].answer == counted_answer
assert re.search(r"(?<!\d)12(?!\d)", counted_answer)
TrajectoryViewer(counting_agent.trajectory)

## URL и форматы изображений

`image_data` принимает:

- обычную HTTP(S)-ссылку — передаётся серверу без загрузки нашим кодом;
- raw base64 для PNG, как в книге;
- полный data URL `data:image/<format>;base64,...` для PNG, JPEG, WebP или GIF.

Проверяется корректность формата передачи, а не декодирование изображения. Для JPEG нужно указать `image/jpeg`; raw base64 автоматически помечается PNG. Реальная поддержка форматов и загрузки URL зависит от inference-сервера. В живых примерах выше проверен **PNG через base64**, не внешние URL.

Следующая ячейка только показывает формат сообщения с URL, ничего не скачивает и не отправляет модели.

In [ ]:
url_memory = MultimodalMemory()
url_memory.add("user", "Describe this image", image_data="https://example.com/image.png")
pprint(url_memory.get_messages())

## Границы реализации

- Изображения требуют явного `MultimodalMemory()`. С обычной памятью `image_data` вызывает ошибку до запроса к модели. Автоматического преобразования SummarizationMemory, TrimmingMemory или RAGMemory нет.
- Одно изображение на один пользовательский запрос; несколько изображений можно передавать последовательными запросами.
- История хранится в RAM и не сжимается. Полное изображение из истории отправляется при следующих генерациях; это увеличивает размер запросов и обработку контекста.
- Ошибка backend передаётся вызывающему коду. Картинка остаётся в памяти, но вымышленный ответ в траекторию не записывается.
- Глава 8 продолжает работать с текстовыми делегированиями: картинки автоматически не передаются специалистам.
- Аудио, видео, ViT/CLIP, Whisper и разные способы соединения энкодеров с LLM обсуждаются в главе теоретически. Код реализует входные изображения; загрузки аудио/видео, генерации изображений и обучения модели здесь нет.

Для своих примеров замените `image_path` на локальный PNG. Тестовая картинка создана нашим `scripts/create_vision_fixture.py` средствами стандартной библиотеки Python. Результаты notebook не сохраняются в Git.